# IMPORT

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# DEFINE FOLDERS

In [2]:
BASE_FOLDER = Path("../data")
STATION_FEATURE_FOLDER = BASE_FOLDER / "02_features" / "station_features"
DATASET_INPUT_FOLDER = BASE_FOLDER / "01_clean" / "datasets"

folders = [DATASET_INPUT_FOLDER, STATION_FEATURE_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# LOAD DATA

In [3]:
df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/final_clean_split_data.csv")

print("(Rows, Columns):", df.shape)
print("Stations:", df["station_code"].nunique())
print(df.columns)

(Rows, Columns): (6705169, 20)
Stations: 1417
Index(['timestamp', 'station_code', 'latitude', 'longitude', 'capacity',
       'total_bikes_now', 'total_bikes_taken', 'temperature', 'wind_speed',
       'wind_direction', 'weather_code', 'is_day', 'hour', 'day_of_week',
       'is_weekend', 'sin_hour', 'cos_hour', 'split', 'total_bikes_lag1',
       'total_bikes_lag2'],
      dtype='object')


# SEPERATE DATA

In [4]:
train_dataset = df[df["split"] == 0].copy(deep=True)
val_dataset   = df[df["split"] == 1].copy(deep=True)
test_dataset  = df[df["split"] == 2].copy(deep=True)

datasets = [train_dataset, val_dataset, test_dataset]

print(f"Train rows: {len(train_dataset)}")
print(f"Val rows:   {len(val_dataset)}")
print(f"Test rows:  {len(test_dataset)}")
datasets = [train_dataset, val_dataset, test_dataset]

Train rows: 4695901
Val rows:   1004615
Test rows:  1004653


# NORMALIZE USAGE

In [5]:
train_dataset["usage_ratio"] = train_dataset["total_bikes_now"] / train_dataset["capacity"]
train_dataset["usage_ratio"] = train_dataset["usage_ratio"].clip(0, 1)
# We normalize data as there are numerous different values for capacity.
# Normalizing it makes it possible to easily compare it to other stations.

# DEFINE FEATURE EXTRACTION PER STATION

In [6]:
def extract_station_features(group):
    features = {}

    # Basic usage(mean, std, min and max)
    features["mean_usage"] = group["usage_ratio"].mean()
    features["std_usage"] = group["usage_ratio"].std()
    features["min_usage"] = group["usage_ratio"].min()
    features["max_usage"] = group["usage_ratio"].max()

    # Temporal behavior(the difference between the usage on weekdays and weekends)
    features["mean_weekend"] = group[group["is_weekend"] == 1]["usage_ratio"].mean()
    features["mean_weekday"] = group[group["is_weekend"] == 0]["usage_ratio"].mean()

    features["day_usage"] = group[group["is_day"] == 1]["usage_ratio"].mean()
    features["night_usage"] = group[group["is_day"] == 0]["usage_ratio"].mean()

    # Daily amplitude(average usage of each hour of the day across all days)
    hourly_mean = group.groupby("hour")["usage_ratio"].mean()
    features["daily_amplitude"] = hourly_mean.max() - hourly_mean.min()

    # Weather correlation
    for col in [
        "temperature",
        "wind_speed",
        "wind_direction",
        "weather_code"
    ]:
        if group[col].nunique() > 1:
            features[f"corr_{col}"] = group["usage_ratio"].corr(group[col])
        else:
            features[f"corr_{col}"] = 0.0

    # Station info (static/spatial)
    features["latitude"] = group["latitude"].iloc[0]
    features["longitude"] = group["longitude"].iloc[0]
    features["capacity"] = group["capacity"].iloc[-1] # To get the latest capacity

    # NO BIKE LAG
    # Lag features are time-dependent series and should not be aggregated at station level.
    # They are computed later during model training.
    # features["total_bikes_lag1"] = group["total_bikes_lag1"]
    # features["total_bikes_lag2"] = group["total_bikes_lag2"]

    return pd.Series(features)


# BUILD STATION FEATURE TABLE

In [7]:
station_features = (
    train_dataset.groupby("station_code", sort=False, group_keys=False)
    .apply(extract_station_features)
    .reset_index()
)

print("Station feature table shape:", station_features.shape)
station_features.head()
# Now we have features for each station.

c:\Users\emirh\miniconda3\envs\dalas\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\emirh\miniconda3\envs\dalas\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Station feature table shape: (1417, 17)


C:\Users\emirh\AppData\Local\Temp\ipykernel_330528\1894814528.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(extract_station_features)


,station_code,mean_usage,std_usage,min_usage,max_usage,mean_weekend,mean_weekday,day_usage,night_usage,daily_amplitude,corr_temperature,corr_wind_speed,corr_wind_direction,corr_weather_code,latitude,longitude,capacity
0,1001,0.602810,0.263373,0.0,1.000000,0.550919,0.623972,0.690731,0.549290,0.375714,0.033189,0.240084,0.132461,0.014957,48.857059,2.341798,17.0
1,1002,0.546886,0.303978,0.0,1.000000,0.479616,0.574320,0.642544,0.488656,0.564801,0.132976,0.190445,0.142048,0.021868,48.857993,2.346917,22.0
2,1006,0.488089,0.227872,0.0,0.939394,0.509422,0.479385,0.438874,0.518062,0.402075,0.013647,0.139952,-0.002916,-0.050898,48.862630,2.349807,33.0
3,1007,0.604008,0.274195,0.0,1.000000,0.546517,0.627464,0.732182,0.525946,0.549732,0.330657,0.358805,0.098165,0.021759,48.863589,2.347779,16.0
4,1008,0.545169,0.288851,0.0,1.000000,0.545215,0.545151,0.663116,0.473371,0.552344,0.266445,0.306384,0.071473,0.036884,48.863627,2.342624,32.0


# CLEAN FEATURE TABLE

In [8]:
stations_before = set(station_features["station_code"])

print(f"Before cleaning: {station_features.shape}")
station_features = station_features.replace([np.inf, -np.inf], np.nan)
station_features = station_features.dropna()
print(f"After cleaning: {station_features.shape}")

stations_after = set(station_features["station_code"])

dropped_stations = stations_before - stations_after

print(f"Dropped stations: {len(dropped_stations)}")
print(dropped_stations)
# Dropping stations in case we have invalid values(inf/NaN) that we have missed(we did not).

Before cleaning: (1417, 17)
After cleaning: (1416, 17)
Dropped stations: 1
{16202}


# SAVE STATION FEATURES

In [9]:
station_features.to_csv(f"{STATION_FEATURE_FOLDER}/station_features.csv", index=False)
print("Saved station_features.csv")

Saved station_features.csv
